[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/21_gradient_clipping.ipynb)

# 🟢 Easy: Gradient Norm Clipping

Implement **gradient norm clipping** — a training stability technique.

### Signature
```python
def clip_grad_norm(parameters, max_norm: float) -> float:
    # Clip gradients in-place so total norm <= max_norm
    # Returns the original (unclipped) total norm
```

### Algorithm
1. Compute total norm: `sqrt(sum(p.grad.norm()^2 for p in parameters))`
2. If total > max_norm: scale all grads by `max_norm / total`
3. Return original total norm

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.9 MB/s eta 0:00:00


In [2]:
import torch

In [43]:
# ✏️ YOUR IMPLEMENTATION HERE

def clip_grad_norm(parameters, max_norm):
  if isinstance(parameters, torch.Tensor):
    parameters = [parameters]

  grads = [p.grad.detach() for p in parameters if p.grad is not None]

  if len(grads) == 0:
    return torch.tensor(0.0)

  device = grads[0].device
  total_norm = torch.stack([g.norm().to(device) ** 2 for g in grads]).sum().sqrt()

  clip_coef = max_norm / (total_norm + 1e-6)
  if clip_coef < 1.0:
    for g in grads:
      g.mul_(clip_coef)
  return total_norm

In [44]:
# 🧪 Debug
p = torch.randn(100, requires_grad=True)
(p * 10).sum().backward()
print('Before:', p.grad.norm().item())
orig = clip_grad_norm([p], max_norm=1.0)
print('After: ', p.grad.norm().item())
print('Original norm:', orig)

Before: 100.0
After:  0.9999999403953552
Original norm: tensor(100.)


In [45]:
# ✅ SUBMIT
from torch_judge import check
check('gradient_clipping')


🧪 Testing: Gradient Norm Clipping (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Clips to max_norm (3.0ms)
  ✅ [2/4] Returns original norm (1.8ms)
  ✅ [3/4] No change when norm < max_norm (0.7ms)
  ✅ [4/4] Preserves direction (1.8ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (7.4ms total)
  Progress saved. Run status() to see your dashboard.

